# Pandas GroupBy, `.agg()`, and Pivot Tables with API Data

This notebook introduces aggregation gradually. We start with simple examples and build toward richer segment-level summary tables.

## Goals

By the end, you will be able to:

1. Pull data from a public API.
2. Use `.groupby()` with one column and multiple columns.
3. Use simple aggregation methods such as `.mean()`, `.sum()`, and `.count()`.
4. Use `.agg()` for one metric, several metrics, and custom named outputs.
5. Use `pivot_table()` to reshape grouped summaries into a readable cross-tab format.

## Dataset

We will use the **REST Countries API**. It gives us country-level data with several quantitative fields, including:

- `population`
- `area`
- number of borders
- number of time zones
- latitude / longitude

API endpoint used in this notebook:

`https://restcountries.com/v3.1/all?fields=name,region,subregion,population,area,latlng,borders,timezones,landlocked`


## 1. Import libraries

In [1]:
import requests
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Get data from the API

The API returns nested JSON. We will flatten only the fields we need into a clean DataFrame.

In [2]:
url = 'https://restcountries.com/v3.1/all?fields=name,region,subregion,population,area,latlng,borders,timezones,landlocked'

response = requests.get(url, timeout=30)
response.raise_for_status() #if there's an error, this gives a description

# what we get will be put into the data list, and the length of the data return the amount of dictionaries  
data = response.json()
len(data)

250

## 3. Convert the API response into a DataFrame

In [ ]:
# .get does takes in a key, returns the value of the key if it's in the dict
# it's the same thing that indexing by a key...but we like .get bc it has a default value or None instead of an error

In [3]:
rows = []

for country in data:
    # not a number is NaN, and with truthiness, python will see if this the list is empty
    latlng = country.get('latlng') or [np.nan, np.nan]
    borders = country.get('borders') or []
    timezones = country.get('timezones') or []

    # add the "gotten" values inside the empty rows list
    rows.append({
        'country': country.get('name', {}).get('common'),
        'region': country.get('region'),
        'subregion': country.get('subregion'),
        'population': country.get('population'),
        'area': country.get('area'),
        'latitude': latlng[0] if len(latlng) > 0 else np.nan,
        'longitude': latlng[1] if len(latlng) > 1 else np.nan,
        'border_count': len(borders),
        'timezone_count': len(timezones),
        'landlocked': country.get('landlocked')
    })

countries = pd.DataFrame(rows)
countries.head()

,country,region,subregion,population,area,latitude,longitude,border_count,timezone_count,landlocked
0,Anguilla,Americas,Caribbean,16010,91.00,18.25,-63.17,0,1,False
1,Guatemala,Americas,Central America,18079810,"108,889.00",15.50,-90.25,4,1,False
2,Gambia,Africa,Western Africa,2422712,"10,689.00",13.47,-16.57,1,1,False
3,Mexico,Americas,North America,130575786,"1,964,375.00",23.00,-102.00,3,3,False
4,Malawi,Africa,Eastern Africa,20734262,"118,484.00",-13.50,34.00,3,1,True


## 4. Basic data check

Before grouping, quickly inspect the columns and missing values.

In [4]:
countries.to_csv("countries.csv", index=False)

In [5]:
countries[['population', 'area', 'border_count', 'timezone_count', 'latitude', 'longitude']].describe()

,population,area,border_count,timezone_count,latitude,longitude
count,250.00,250.00,250.00,250.00,250.00,250.00
mean,"32,077,981.84","601,038.85",2.59,1.36,16.41,14.15
std,"131,965,523.43","1,912,575.36",2.69,1.51,26.98,74.20
min,0.00,0.49,0.00,1.00,-90.00,-176.20
25%,"223,354.25","1,194.25",0.00,1.00,1.00,-49.75
50%,"5,279,123.00","64,929.50",2.00,1.00,16.39,17.50
75%,"20,371,664.25","384,150.50",4.00,1.00,39.00,50.16
max,"1,417,492,000.00","17,098,246.00",16.00,14.00,78.00,178.06


# Part A: Very simple aggregation examples

Before using `.agg()`, it helps to understand the simplest form of grouping.

The pattern is:

```python
dataframe.groupby('category_column')['numeric_column'].summary_method()
```


## 5. Count how many countries are in each region

This uses `.size()` because we want to count rows per group.

In [ ]:
countries.groupby('region').size()

To make the result easier to read, convert it back into a DataFrame.

In [ ]:
region_counts = (
    countries
    .groupby('region')
    .size()
    .reset_index(name='country_count')
    .sort_values('country_count', ascending=False)
)

region_counts

## 6. Average population by region

This is one numeric column and one aggregation method.

In [ ]:
countries.groupby('region')['population'].mean()

Again, we can make it cleaner with `reset_index()` and sorting.

In [ ]:
avg_population_by_region = (
    countries
    .groupby('region')['population']
    .mean()
    .reset_index(name='avg_population')
    .sort_values('avg_population', ascending=False)
)

avg_population_by_region

## 7. Total population by region

Same pattern, different aggregation method.

In [ ]:
total_population_by_region = (
    countries
    .groupby('region')['population']
    .sum()
    .reset_index(name='total_population')
    .sort_values('total_population', ascending=False)
)

total_population_by_region

# Part B: Introducing `.agg()` slowly

`.agg()` lets you specify aggregation functions explicitly.

Start with a single column and a single function.

## 8. `.agg()` with one column and one function

In [ ]:
countries.groupby('region')['population'].agg('mean')

This produces the same result as `.mean()`, but `.agg()` becomes more useful as the summary gets more complex.

In [ ]:
countries.groupby('region')['population'].mean()

## 9. `.agg()` with one column and multiple functions

Now summarize population with several statistics at once.

In [ ]:
population_summary_simple = (
    countries
    .groupby('region')['population']
    .agg(['count', 'sum', 'mean', 'median', 'min', 'max'])
    .reset_index()
)

population_summary_simple

## 10. Rename columns after a simple `.agg()`

The output column names are the function names. You can rename them to make the table easier to understand.

In [ ]:
population_summary_simple = population_summary_simple.rename(columns={
    'count': 'country_count',
    'sum': 'total_population',
    'mean': 'avg_population',
    'median': 'median_population',
    'min': 'smallest_population',
    'max': 'largest_population'
})

population_summary_simple

# Part C: `.agg()` with multiple numeric columns

Now we will summarize more than one quantitative field.

## 11. Dictionary style: column → aggregation functions

This says:

- For `population`, calculate `sum` and `mean`
- For `area`, calculate `sum` and `mean`
- For `border_count`, calculate `mean` and `max`


In [ ]:
region_summary_dict = (
    countries
    .groupby('region')
    .agg({
        'population': ['sum', 'mean'],
        'area': ['sum', 'mean'],
        'border_count': ['mean', 'max']
    })
)

region_summary_dict

This works, but it creates **multi-level column names**. That can be useful, but it is often less convenient for beginners.

In [ ]:
region_summary_dict.columns

## 12. Flatten multi-level columns

This converts column names like `('population', 'sum')` into `population_sum`.

In [ ]:
region_summary_dict_flat = region_summary_dict.copy()

region_summary_dict_flat.columns = [
    '_'.join(col).strip('_')
    for col in region_summary_dict_flat.columns
]

region_summary_dict_flat = region_summary_dict_flat.reset_index()

region_summary_dict_flat

# Part D: Named aggregation

Named aggregation is usually easier to read because you choose the output column names directly.

The pattern is:

```python
.agg(
    new_column_name=('existing_column', 'aggregation_function')
)
```


## 13. Named aggregation with clear output names

In [ ]:
region_summary = (
    countries
    .groupby('region')
    .agg(
        country_count=('country', 'count'),
        total_population=('population', 'sum'),
        avg_population=('population', 'mean'),
        median_population=('population', 'median'),
        total_area=('area', 'sum'),
        avg_area=('area', 'mean'),
        avg_border_count=('border_count', 'mean'),
        max_border_count=('border_count', 'max'),
        avg_timezone_count=('timezone_count', 'mean')
    )
    .reset_index()
    .sort_values('total_population', ascending=False)
)

region_summary

## 14. Add a calculated field after aggregation

Because `region_summary` is a normal DataFrame after `reset_index()`, we can create new columns.

In [ ]:
region_summary['population_density'] = (
    region_summary['total_population'] / region_summary['total_area']
)

region_summary.sort_values('population_density', ascending=False)

# Part E: Grouping by multiple columns

Now we move from region-level summaries to segment-level summaries.

Here, a segment is a combination of:

- `region`
- `landlocked`

This answers questions like:

> Within each region, how do landlocked and non-landlocked countries differ?


## 15. Basic multi-column group count

In [ ]:
countries.groupby(['region', 'landlocked']).size()

Make it a normal DataFrame.

In [ ]:
region_landlocked_counts = (
    countries
    .groupby(['region', 'landlocked'])
    .size()
    .reset_index(name='country_count')
    .sort_values(['region', 'landlocked'])
)

region_landlocked_counts

## 16. Multi-column groupby with named aggregation

In [ ]:
segment_summary = (
    countries
    .groupby(['region', 'landlocked'])
    .agg(
        country_count=('country', 'count'),
        total_population=('population', 'sum'),
        avg_population=('population', 'mean'),
        median_population=('population', 'median'),
        total_area=('area', 'sum'),
        avg_area=('area', 'mean'),
        avg_border_count=('border_count', 'mean'),
        avg_timezone_count=('timezone_count', 'mean')
    )
    .reset_index()
    .sort_values('total_population', ascending=False)
)

segment_summary

## 17. Another multi-column grouping: region and subregion

This creates a more detailed segment-level summary.

In [ ]:
subregion_summary = (
    countries
    .dropna(subset=['subregion'])
    .groupby(['region', 'subregion'])
    .agg(
        country_count=('country', 'count'),
        total_population=('population', 'sum'),
        avg_population=('population', 'mean'),
        total_area=('area', 'sum'),
        avg_area=('area', 'mean'),
        avg_border_count=('border_count', 'mean')
    )
    .reset_index()
    .sort_values('total_population', ascending=False)
)

subregion_summary.head(15)

# Part F: Pivot tables

A pivot table reshapes grouped data into a cross-tabular format.

The basic pattern is:

```python
pd.pivot_table(
    data=...,
    index='row_category',
    columns='column_category',
    values='numeric_field',
    aggfunc='summary_function'
)
```


## 18. Pivot table: country count by region and landlocked status

In [ ]:
landlocked_count_pivot = pd.pivot_table(
    data=countries,
    index='region',
    columns='landlocked',
    values='country',
    aggfunc='count',
    fill_value=0
)

landlocked_count_pivot

## 19. Clean up pivot column names

The columns are currently Boolean values: `False` and `True`. We can rename them.

In [ ]:
landlocked_count_pivot = landlocked_count_pivot.rename(columns={
    False: 'not_landlocked',
    True: 'landlocked'
})

landlocked_count_pivot

## 20. Pivot table: average population by region and landlocked status

In [ ]:
avg_population_pivot = pd.pivot_table(
    data=countries,
    index='region',
    columns='landlocked',
    values='population',
    aggfunc='mean',
    fill_value=0
).rename(columns={
    False: 'not_landlocked',
    True: 'landlocked'
})

avg_population_pivot

## 21. Pivot table with multiple aggregation functions

This produces a multi-level column structure, similar to dictionary-style `.agg()`.

In [ ]:
population_pivot_multiagg = pd.pivot_table(
    data=countries,
    index='region',
    columns='landlocked',
    values='population',
    aggfunc=['count', 'sum', 'mean'],
    fill_value=0
)

population_pivot_multiagg

## 22. Pivot table with multiple values

Now summarize both `population` and `area`.

In [ ]:
multi_value_pivot = pd.pivot_table(
    data=countries,
    index='region',
    columns='landlocked',
    values=['population', 'area'],
    aggfunc='mean',
    fill_value=0
)

multi_value_pivot

# Part G: Practice exercises

Try these on your own. Suggested answers are included below each prompt.

## Exercise 1

Create a summary showing the **average area** by region.

In [ ]:
# Your code here

In [ ]:
# Suggested answer
area_by_region = (
    countries
    .groupby('region')['area']
    .mean()
    .reset_index(name='avg_area')
    .sort_values('avg_area', ascending=False)
)

area_by_region

## Exercise 2

Create a summary by region with:

- country count
- total population
- average population
- average area


In [ ]:
# Your code here

In [ ]:
# Suggested answer
exercise_2 = (
    countries
    .groupby('region')
    .agg(
        country_count=('country', 'count'),
        total_population=('population', 'sum'),
        avg_population=('population', 'mean'),
        avg_area=('area', 'mean')
    )
    .reset_index()
    .sort_values('total_population', ascending=False)
)

exercise_2

## Exercise 3

Create a segment-level summary grouped by `region` and `landlocked` with:

- country count
- total population
- average border count


In [ ]:
# Your code here

In [ ]:
# Suggested answer
exercise_3 = (
    countries
    .groupby(['region', 'landlocked'])
    .agg(
        country_count=('country', 'count'),
        total_population=('population', 'sum'),
        avg_border_count=('border_count', 'mean')
    )
    .reset_index()
    .sort_values('total_population', ascending=False)
)

exercise_3

## Exercise 4

Create a pivot table showing average `border_count` by region and landlocked status.

In [ ]:
# Your code here

In [ ]:
# Suggested answer
exercise_4 = pd.pivot_table(
    data=countries,
    index='region',
    columns='landlocked',
    values='border_count',
    aggfunc='mean',
    fill_value=0
).rename(columns={
    False: 'not_landlocked',
    True: 'landlocked'
})

exercise_4

# Key takeaways

- Use `.groupby('column')` to summarize by one category.
- Use `.groupby(['col1', 'col2'])` to summarize by combinations of categories.
- Use `.size()` for row counts.
- Use `.agg()` when you want multiple summary metrics.
- Use named aggregation when you want clean, custom output column names.
- Use `pivot_table()` when you want grouped results reshaped into rows and columns.
